# exp-h0-component-complement-audit-01

H0 내부의 Selective-EB, non-EB, EB 및 동일 specialist 확률이 서로 보완되는지 보는 마지막 소폭 앙상블 감사입니다. 고정 0.5 평균만 비교하며 가중치 탐색·제출 생성은 하지 않습니다.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
BASE = ROOT / 'experiments' / 'gs' / 'notebooks' / 'exp_model_011'
RUNNER = BASE / 'common' / 'run_h0_component_complement_audit.py'
RESULT = BASE / 'result'
RUN_ID = 'exp-h0-component-complement-audit-01'
RUN_EXPERIMENT = False
assert (ROOT / 'data' / 'raw' / 'train.csv').exists()
assert RUNNER.exists()
print({'runner': RUNNER, 'test_read': False, 'fixed_blends': ['0.5/0.5']})

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='H0 component audit', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-100:]
    if process.wait() != 0:
        raise RuntimeError('component audit runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 기존 결과만 읽습니다.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(RESULT / f'{RUN_ID}_seed42_summary.csv')
folds = pd.read_csv(RESULT / f'{RUN_ID}_seed42_fold_metrics.csv')
overlap = pd.read_csv(RESULT / f'{RUN_ID}_seed42_error_overlap.csv')
audit = json.loads((RESULT / f'{RUN_ID}_seed42_audit.json').read_text())
assert summary.leakage_check.all()
assert summary.nan_as_mutation_count.eq(0).all()
assert audit['test_read'] is False and audit['leakage_check'] is True
display(summary.sort_values('oof_macro_f1', ascending=False))
display(overlap.sort_values('h0_wrong_recovered', ascending=False))
audit

In [ ]:
folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', figsize=(9, 4), title='H0 internal fixed-combination audit')
plt.ylabel('Macro F1'); plt.tight_layout(); plt.show()
best = summary.loc[summary.variant.ne('H0_selective_EB')].sort_values('oof_macro_f1', ascending=False).iloc[0]
print(f"최고 fixed 후보: {best.variant}, Δ={best.delta_vs_h0:+.6f}")
print('3-seed 후보:', audit['three_seed_candidate'])